### np.matmul

In [1]:
def get_shape(a):
    if not isinstance(a, list):
        return ()
    if len(a) == 0:
        return (0,)
    inner_shape = get_shape(a[0])
    for i in range(1, len(a)):
        if get_shape(a[i]) != inner_shape:
            raise ValueError(
                f"Jagged array: first row shape {inner_shape}, "
                f"but row {i} shape {get_shape(a[i])}"
            )
    return (len(a),) + inner_shape


def _product(shape):
    p = 1
    for s in shape:
        p *= s
    return p


def _get_item(a, idx):
    cur = a
    for i in idx:
        cur = cur[i]
    return cur


def _build_nested(shape, getter, prefix=()):
    if len(shape) == 0:
        return getter(prefix)
    return [_build_nested(shape[1:], getter, prefix + (i,)) for i in range(shape[0])]


def np_matmul(a, b):

    shape_a = get_shape(a)
    shape_b = get_shape(b)
    ndim_a = len(shape_a)
    ndim_b = len(shape_b)

    if ndim_a == 0 or ndim_b == 0:
        raise ValueError("scalars not allowed in matmul")

   
    if ndim_a == 1 and ndim_b == 1:
        if shape_a[0] != shape_b[0]:
            raise ValueError("shapes not aligned for 1-D matmul")
        total = 0
        for i in range(shape_a[0]):
            total += a[i] * b[i]
        return total

    if ndim_a == 2 and ndim_b == 2:
        if shape_a[1] != shape_b[0]:
            raise ValueError("shapes not aligned for matrix multiplication")
        m, k = shape_a[0], shape_a[1]
        k, n = shape_b[0], shape_b[1]
        out_shape = (m, n)

        def getter(idx):
            i, j = idx
            total = 0
            for c in range(k):
                total += a[i][c] * b[c][j]
            return total

        return _build_nested(out_shape, getter)

    if ndim_a == 2 and ndim_b == 1:
        if shape_a[1] != shape_b[0]:
            raise ValueError("shapes not aligned for 2-D × 1-D matmul")
        m, k = shape_a[0], shape_a[1]
        out_shape = (m,)

        def getter(idx):
            i = idx[0]
            total = 0
            for j in range(k):
                total += a[i][j] * b[j]
            return total

        return _build_nested(out_shape, getter)

    if ndim_a == 1 and ndim_b == 2:
        if shape_a[0] != shape_b[0]:
            raise ValueError("shapes not aligned for 1-D × 2-D matmul")
        k, n = shape_b[0], shape_b[1]
        out_shape = (n,)

        def getter(idx):
            j = idx[0]
            total = 0
            for i in range(k):
                total += a[i] * b[i][j]
            return total

        return _build_nested(out_shape, getter)

    if ndim_a < 2 or ndim_b < 2:
        raise ValueError("N‑D inputs must have at least 2 dimensions")

    if shape_a[-1] != shape_b[-2]:
        raise ValueError("shapes not aligned: last axis of a must match second-to-last of b")

    leading_a = shape_a[:-2]
    leading_b = shape_b[:-2]

    if len(leading_a) == 0 and len(leading_b) == 0:
        leading_out = ()
    elif len(leading_a) == 0:
        leading_out = leading_b
    elif len(leading_b) == 0:
        leading_out = leading_a
    else:
        if len(leading_a) != len(leading_b):
            raise ValueError("batch shapes must have same number of dims")
        leading_out = []
        for i, d1, d2 in zip(range(len(leading_a)), leading_a, leading_b):
            if d1 == d2:
                leading_out.append(d1)
            elif d1 == 1:
                leading_out.append(d2)
            elif d2 == 1:
                leading_out.append(d1)
            else:
                raise ValueError(f"dimensions {d1} and {d2} at axis {i} are not broadcastable")
        leading_out = tuple(leading_out)

    k = shape_a[-2]  
    m = shape_a[-1] 
    n = shape_b[-1]  

    out_shape = leading_out + (k, n)

    def getter(out_idx):

        lead_idx = out_idx[:len(leading_out)]
        i = out_idx[len(leading_out)]
        j = out_idx[len(leading_out) + 1]


        def map_idx(lead_shape, idx):
            if len(lead_shape) == 0:
                return ()
            return tuple(c if lead_shape[i] > 1 else 0 for i, c in enumerate(idx))

        a_lead = map_idx(leading_a, lead_idx)
        b_lead = map_idx(leading_b, lead_idx)

        total = 0
        for c in range(m):
            idx_a = a_lead + (i, c)
            idx_b = b_lead + (c, j)
            total += _get_item(a, idx_a) * _get_item(b, idx_b)

        return total

    return _build_nested(out_shape, getter)

### test cases 

In [3]:
print(np_matmul([1, 2, 3], [4, 5, 6]))
print(np_matmul([[1, 2], [3, 4]], [[5, 6], [7, 8]]))
print(np_matmul([[1, 2], [3, 4]], [5, 6]))
print(np_matmul([1, 2], [[5, 6], [7, 8]]))
print(np_matmul([[[1, 2], [3, 4]]], [[[5, 6], [7, 8]]]))
print(np_matmul([[[1, 2]], [[3, 4]]], [[[5], [6]], [[7], [8]]]))

32
[[19, 22], [43, 50]]
[17, 39]
[19, 22]
[[[19, 22], [43, 50]]]
[[[17]], [[53]]]


In [4]:
print(np_matmul(2, 3))

ValueError: scalars not allowed in matmul

In [5]:
print(py_matmul([1, 2], [1, 2, 3]))

NameError: name 'py_matmul' is not defined

In [6]:
print(py_matmul([[1, 2, 3]], [4, 5, 6, 7]))

NameError: name 'py_matmul' is not defined

In [7]:
print(py_matmul([[[1, 2], [3, 4]], [[5, 6]]], [[[7, 8], [9, 0]]]))

NameError: name 'py_matmul' is not defined

In [8]:
print(py_matmul([], [1, 2, 3]))

NameError: name 'py_matmul' is not defined